### Plot the result

this is the plot notebook from the official [CAFA repo](https://github.com/BioComputingUP/CAFA-evaluator/blob/main/src/plot.ipynb)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
plt.set_loglevel("info")

#### Config

In [ ]:
# Set to None if you don't want to use it. Results will not be grouped/filtered by team
names_file = None
# Cumulate the last column of the cols variable, e.g. "pr" --> precision, so that the curves are monotonic as in CAFA
cumulate = True
# Add extreme points to the precision-recall curves (0, 1) and (1, 0)
add_extreme_points = True
# Methods with coverage below this threshold will not be plotted
coverage_threshold = 0.3
# Select a metric
metric, cols = ('f', ['rc', 'pr'])

# Map column names to full names (for axis labels)
axis_title_dict = {'pr': 'Precision',
                    'rc': 'Recall',
                    'f': 'F-score',
                    'pr_w': 'Weighted Precision',
                    'rc_w': 'Weighted Recall',
                    'f_w': 'Weighted F-score',
                    'mi': 'Misinformation (Unweighted)',
                    'ru': 'Remaining Uncertainty (Unweighted)',
                    'mi_w': 'Misinformation',
                    'ru_w': 'Remaining Uncertainty',
                    's': 'S-score',
                    'pr_micro': 'Precision (Micro)',
                    'rc_micro': 'Recall (Micro)',
                    'f_micro': 'F-score (Micro)',
                    'pr_micro_w': 'Weighted Precision (Micro)',
                    'rc_micro_w': 'Weighted Recall (Micro)',
                    'f_micro_w': 'Weighted F-score (Micro)'}

# Map ontology namespaces to full names (for plot titles)
ontology_dict = {'biological_process': 'BPO',
                    'molecular_function': 'MFO',
                    'cellular_component': 'CCO'}

#### I/O path

In [ ]:
# Input files
df_file = "output/example1/evaluation_all.tsv"
out_folder = "plots/example1"
os.makedirs(out_folder, exist_ok=True)

#### Load data

In [ ]:
df = pd.read_csv(df_file, sep="\t")
print(df.shape)
display(df.head(2))

#### Add group/label/color

In [ ]:
# Set method information (optional)
if names_file is None:
    df['group'] = df['filename']
    df['label'] = df['filename']
    df['is_baseline'] = False
else:
    methods = pd.read_csv(names_file, delim_whitespace=True, header=0)
    df = pd.merge(df, methods, on='filename', how='left')
    df['group'].fillna(df['filename'], inplace=True)
    df['label'].fillna(df['filename'], inplace=True)
    if 'is_baseline' not in df:
        df['is_baseline'] = False
    else:
        df['is_baseline'].fillna(False, inplace=True)
    # print(methods)
df = df.drop(columns='filename').set_index(['group', 'label', 'ns', 'tau'])
# Filter by coverage
df = df[df['cov'] >= coverage_threshold]
# Assign colors based on group
cmap = plt.get_cmap('tab20')
df['colors'] = df.index.get_level_values('group')
df['colors'] = pd.factorize(df['colors'])[0]
df['colors'] = df['colors'].apply(lambda x: cmap.colors[x % len(cmap.colors)])

#### Identify the best methods and thresholds

In [ ]:
if metric in ['f', 'f_w', 'f_micro', 'f_micro_w']:
    index_best = df.groupby(level=['group', 'ns'])[metric].idxmax()
else: index_best = df.groupby(['group', 'ns'])[metric].idxmin()
display(index_best)

#### Create and persist plot dataset

In [ ]:
# Filter the dataframe for the best methods
df_methods = df.reset_index('tau').loc[
    [ele[:-1] for ele in index_best],
    ['tau', 'cov', 'colors'] + cols + [metric]
].sort_index()

# Makes the curves monotonic. Cumulative max on the last column of the cols variable, e.g. "pr" --> precision
if cumulate:
    if metric in ['f', 'f_w', 'f_micro', 'f_micro_w']:
        df_methods[cols[-1]] = df_methods.groupby(level=['label', 'ns'])[cols[-1]].cummax()
    else:
        df_methods[cols[-1]] = df_methods.groupby(level=['label', 'ns'])[cols[-1]].cummin()

# Save to file
df_methods.drop(columns=['colors'])\
    .to_csv('{}/fig_{}.tsv'.format(out_folder, metric), float_format="%.3f", sep="\t")

#### Create legend + avg precision score(aps)

In [ ]:
# Filter the dataframe for the best method and threshold
df_best = df.loc[index_best, ['cov', 'colors'] + cols + [metric]]

# Calculate average precision score 
if metric.startswith('f'):
    df_best['aps'] = df_methods\
        .groupby(level=['group', 'label', 'ns'])[[cols[0], cols[1]]]\
        .apply(lambda x: (x[cols[0]].diff(-1).shift(1) * x[cols[1]]).sum())

# Calculate the max coverage across all thresholds
df_best['max_cov'] = df_methods.groupby(level=['group', 'label', 'ns'])['cov'].max()

# Set a label column for the plot legend
df_best['label'] = df_best.index.get_level_values('label')
if 'aps' not in df_best.columns:
    df_best['label'] = df_best\
        .agg(lambda x: f"{x['label']} ({metric.upper()}={x[metric]:.3f} C={x['max_cov']:.3f})",
             axis=1)
else:
    df_best['label'] = df_best\
        .agg(lambda x: f"{x['label']} ({metric.upper()}={x[metric]:.3f} APS={x['aps']:.3f} C={x['max_cov']:.3f})",
             axis=1)

#### Plot

In [ ]:
# Generate the figures
plt.rcParams.update({'font.size': 22, 'legend.fontsize': 18})

# F-score contour lines
x = np.arange(0.01, 1, 0.01)
y = np.arange(0.01, 1, 0.01)
X, Y = np.meshgrid(x, y)
Z = 2 * X * Y / (X + Y)

for ns, df_g in df_best.groupby(level='ns'):
    fig, ax = plt.subplots(figsize=(15, 15))

     # Contour lines. At the moment they are provided only for the F-score
    if metric.startswith('f'):
        CS = ax.contour(X, Y, Z, np.arange(0.1, 1.0, 0.1), colors='gray')
        ax.clabel(CS, inline=True) #, fontsize=10)

    # Iterate methods
    for i, (index, row) in enumerate(df_g.sort_values(by=[metric, 'max_cov'], ascending=[False if metric.startswith('f') else True, False]).iterrows()):
        data = df_methods.loc[index[:-1]]
        
        # Precision-recall or mi-ru curves
        ax.plot(data[cols[0]], data[cols[1]], color=row['colors'], label=row['label'], lw=2, zorder=500-i)
        
        # F-max or S-min dots
        ax.plot(row[cols[0]], row[cols[1]], color=row['colors'], marker='o', markersize=12, mfc='none', zorder=1000-i)
        ax.plot(row[cols[0]], row[cols[1]], color=row['colors'], marker='o', markersize=6, zorder=1000-i)

    # Set axes limit
    if metric.startswith('f'):
        plt.xlim(0, 1)
        plt.ylim(0, 1)

    # Set titles
    ax.set_title(ontology_dict.get(ns, ns), pad=20)
    ax.set_xlabel(axis_title_dict[cols[0]], labelpad=20)
    ax.set_ylabel(axis_title_dict[cols[1]], labelpad=20)
    
    leg = ax.legend(markerscale=6)
    for legobj in leg.get_lines():
        legobj.set_linewidth(10.0)

    # Save figure on disk
    plt.savefig("{}/fig_{}_{}.png".format(out_folder, metric, ns), bbox_inches='tight', dpi=300, transparent=False)
    plt.clf()